In [3]:
import tifffile
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import cv2
import pandas as pd
import seaborn as sns
from skimage.measure import regionprops, label
from scipy.signal import savgol_filter, find_peaks, peak_widths
import glob

In [4]:
def order_box_points(box):
    s = box.sum(axis=1)
    diff = np.diff(box, axis=1).flatten()
    ordered = np.zeros((4, 2), dtype=np.float32)
    ordered[0] = box[np.argmin(s)]      # top-left
    ordered[2] = box[np.argmax(s)]      # bottom-right
    ordered[1] = box[np.argmin(diff)]   # top-right
    ordered[3] = box[np.argmax(diff)]   # bottom-left
    return ordered

## Part 1: Batch process all tifs and save perfusion profiles as CSVs

For each tif: rectify the chip against its minimum-area bounding rectangle, sum the dextran intensity across the chip's width for every position along its long axis (y), and save the resulting profile as a CSV. The masked/rectified dextran image is also saved (as `.npy`) so Part 2 can build the paired image + profile figure without reprocessing the raw tifs.

In [5]:
tifs = glob.glob(r"z:\Bel\Farid\Paired_Dextran_Images\vm_outputs\*8channel.tif")
print(f"Found {len(tifs)} tif files")

Found 96 tif files


In [6]:
def process_tif(tif_path):
    """Load a single 8-channel tif, rectify the chip against its minimum-area
    bounding rectangle, and return the masked/rectified dextran intensity image
    plus the intensity profile summed across the chip's width for every position
    along its long axis (y).

    The rectified image always has its long axis along rows (axis 0), so the
    intensity profile (summed across axis 1) always runs along the chip's
    length, regardless of how the chip happens to be oriented in the raw image.
    """
    stack = tifffile.imread(tif_path)

    dextran = stack[:, 2, :, :]
    full_segmentation = stack[:, -4, :, :]
    valid_region = stack[:, 3, :, :]

    valid_dextran = dextran * (valid_region > 0) * (full_segmentation > 0)
    chip_2d = np.max(valid_region, axis=0)
    flat_dextran = np.sum(valid_dextran, axis=0)

    contours, _ = cv2.findContours(
        (chip_2d > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    largest = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(largest)  # ((cx, cy), (w, h), angle)
    box = cv2.boxPoints(rect)

    src = order_box_points(box)  # tl, tr, br, bl (image-space order)

    # Measure the two side lengths directly from the ordered corners, since
    # cv2.minAreaRect's own (w, h) order isn't guaranteed to stay consistent
    # with the chip's actual long axis across different rotation angles.
    edge_tl_tr = np.linalg.norm(src[1] - src[0])
    edge_tr_br = np.linalg.norm(src[2] - src[1])

    if edge_tl_tr >= edge_tr_br:
        # tl -> tr runs along the long axis; put length along rows (y) and
        # width along columns (x) by rotating the destination corner mapping.
        object_length = int(round(edge_tl_tr))
        object_width = int(round(edge_tr_br))
        dst = np.array([
            [0, 0],
            [0, object_length - 1],
            [object_width - 1, object_length - 1],
            [object_width - 1, 0]
        ], dtype=np.float32)
    else:
        # tr -> br runs along the long axis; this is already rows-as-length.
        object_length = int(round(edge_tr_br))
        object_width = int(round(edge_tl_tr))
        dst = np.array([
            [0, 0],
            [object_width - 1, 0],
            [object_width - 1, object_length - 1],
            [0, object_length - 1]
        ], dtype=np.float32)

    M = cv2.getPerspectiveTransform(src, dst)
    out_size = (object_width, object_length)  # (width, height) for warpPerspective

    img_rectified = cv2.warpPerspective(
        flat_dextran.astype(np.float32), M, out_size, flags=cv2.INTER_LINEAR
    )
    mask_rectified = cv2.warpPerspective(
        chip_2d, M, out_size, flags=cv2.INTER_NEAREST
    ).astype(bool)

    img_masked = np.where(mask_rectified, img_rectified, 0)

    # Sum across columns (width) for every row (position along the long axis)
    sums = img_masked.sum(axis=1)
    ys = np.arange(sums.shape[0])
    profile_df = pd.DataFrame({"y": ys, "intensity": sums})

    return profile_df, img_masked

In [ ]:
output_dir = Path(r"Z:\Bel\Farid\Paired_Dextran_Images\perfusion_profiles")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Processing {len(tifs)} tif files -> {output_dir}")

for tif_path in tifs:
    name = Path(tif_path).stem
    try:
        profile_df, img_masked = process_tif(tif_path)
        profile_df.to_csv(output_dir / f"{name}_profile.csv", index=False)
        np.save(output_dir / f"{name}_image.npy", img_masked)
        print(f"  OK   {name} ({len(profile_df)} rows)")
    except Exception as e:
        print(f"  FAIL {name}: {e}")

Processing 96 tif files -> Z:\Bel\Farid\Paired_Dextran_Images\perfusion_profiles
  OK   010826_N3_flow_day_10_R_1_Merged_8channel (5920 rows)
  OK   010826_N3_flow_day_10_R_2_Merged_8channel (5976 rows)
  OK   010826_N3_flow_day_10_R_3_Merged_8channel (5976 rows)
  OK   010826_N3_flow_day_10_R_4_Merged_8channel (5656 rows)


## Part 2: Load saved profiles, smooth, extract heterogeneity statistics, and visualize

Reads the CSVs saved by Part 1, applies the same rolling-mean + Savitzky-Golay smoothing used previously, and computes shape-agnostic heterogeneity metrics (coefficient of variation, Gini coefficient, peak count/width/spacing variability) for every chip. The final cell builds a 2-axis figure pairing the rectified dextran image (rotated so the chip's long axis runs left-to-right) with its intensity profile.

In [ ]:
def compute_profile_stats(profile_df, roll_window=500, savgol_window=1000, savgol_poly=2,
                            peak_prominence=5, peak_distance=5):
    """Smooth a raw intensity profile and compute heterogeneity statistics that
    don't assume a single-peak (bell curve) shape."""
    smoothed = (
        profile_df
        .rolling(window=roll_window, on="intensity")
        .mean(numeric_only=True)
        .reset_index()
    )
    smoothed = smoothed[smoothed["y"] > 0].reset_index(drop=True)

    if len(smoothed) <= savgol_window:
        smoothed["intensity_savgol"] = smoothed["intensity"]
    else:
        smoothed["intensity_savgol"] = savgol_filter(
            smoothed["intensity"], window_length=savgol_window, polyorder=savgol_poly
        )

    signal = smoothed["intensity_savgol"].to_numpy()
    y = smoothed["y"].to_numpy()
    valid = signal[~np.isnan(signal)]

    cv = valid.std() / valid.mean() if valid.mean() != 0 else np.nan

    def gini(x):
        x = np.sort(x[x >= 0])
        n = len(x)
        if n == 0 or x.sum() == 0:
            return np.nan
        cumx = np.cumsum(x)
        return (n + 1 - 2 * np.sum(cumx) / cumx[-1]) / n

    gini_coef = gini(valid)

    peaks, properties = find_peaks(signal, prominence=peak_prominence, distance=peak_distance)
    if len(peaks) > 0:
        widths = peak_widths(signal, peaks, rel_height=0.5)[0]
        width_cv = widths.std() / widths.mean() if widths.mean() != 0 else np.nan
        spacing = np.diff(y[peaks]) if len(peaks) > 1 else np.array([])
        spacing_cv = spacing.std() / spacing.mean() if len(spacing) > 0 and spacing.mean() != 0 else np.nan
        mean_prominence = properties["prominences"].mean()
    else:
        width_cv = np.nan
        spacing_cv = np.nan
        mean_prominence = np.nan

    stats = {
        "cv": cv,
        "gini": gini_coef,
        "n_peaks": len(peaks),
        "peak_width_cv": width_cv,
        "peak_spacing_cv": spacing_cv,
        "mean_peak_prominence": mean_prominence,
    }
    return stats, smoothed, peaks, properties

In [ ]:
profile_csvs = sorted(glob.glob(str(output_dir / "*_profile.csv")))
print(f"Found {len(profile_csvs)} saved profiles")

records = []
for csv_path in profile_csvs:
    name = Path(csv_path).stem.replace("_profile", "")
    profile_df = pd.read_csv(csv_path)
    stats, _, _, _ = compute_profile_stats(profile_df)
    stats["name"] = name
    records.append(stats)

stats_df = pd.DataFrame(records).set_index("name")
stats_df

In [ ]:
sample_name = stats_df.index[0]  # change this to inspect a different chip

profile_df = pd.read_csv(output_dir / f"{sample_name}_profile.csv")
img_masked = np.load(output_dir / f"{sample_name}_image.npy")
stats, smoothed, peaks, properties = compute_profile_stats(profile_df)

# Rotate 90 degrees (transpose) so the chip's long axis (y) runs left-to-right,
# matching the profile plot below (y=0 on the left)
rotated_image = img_masked.T

fig, ax = plt.subplots(
    2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [1, 2]}
)

ax[0].imshow(
    rotated_image, cmap="gray", aspect="auto",
    extent=[0, rotated_image.shape[1], 0, rotated_image.shape[0]]
)
ax[0].set_yticks([])
ax[0].set_title(f"{sample_name} - dextran (rectified)")

sns.lineplot(data=smoothed, x="y", y="intensity", ax=ax[1], label="Rolling mean", alpha=0.5)
sns.lineplot(data=smoothed, x="y", y="intensity_savgol", ax=ax[1], label="Savgol")
ax[1].plot(
    smoothed["y"].to_numpy()[peaks], smoothed["intensity_savgol"].to_numpy()[peaks],
    "rx", label="Peaks"
)
ax[1].set_xlabel("y (position along chip length)")
ax[1].set_ylabel("Summed dextran intensity")
ax[1].set_title(
    f"CV={stats['cv']:.2f}  Gini={stats['gini']:.2f}  n_peaks={stats['n_peaks']}"
)
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
from vtkmodules.vtkFiltersGeneral import (
    vtkDiscreteFlyingEdges3D,
    vtkDiscreteMarchingCubes
)


In [ ]:
import numpy as np
import vtk
from vtk.util.numpy_support import numpy_to_vtk, vtk_to_numpy
import napari


seg = (full_segmentation > 0).astype(np.uint8)

# ------------------------------------------------------------
# NumPy -> vtkImageData
# ------------------------------------------------------------

z, y, x = seg.shape

vtk_image = vtk.vtkImageData()
vtk_image.SetDimensions(x, y, z)

# VTK coordinates are X, Y, Z
# Your napari scale is Z, Y, X = (10, 2, 2)
vtk_image.SetSpacing(2, 2, 10)

vtk_array = numpy_to_vtk(
    seg.ravel(order="C"),
    deep=True,
    array_type=vtk.VTK_UNSIGNED_CHAR,
)

vtk_image.GetPointData().SetScalars(vtk_array)


# ------------------------------------------------------------
# Surface Nets
# ------------------------------------------------------------

surface_nets = vtk.vtkSurfaceNets3D()
surface_nets.SetInputData(vtk_image)

# Extract label 1
surface_nets.SetValue(0, 1)

# Built-in smoothing
surface_nets.SetSmoothing(True)
surface_nets.SetNumberOfIterations(500)

# Napari needs triangles
surface_nets.SetOutputMeshTypeToTriangles()

surface_nets.Update()

mesh = surface_nets.GetOutput()


# ------------------------------------------------------------
# VTK mesh -> NumPy
# ------------------------------------------------------------

# VTK vertices are X, Y, Z
vertices_xyz = vtk_to_numpy(mesh.GetPoints().GetData())

# Napari volume coordinates are Z, Y, X
vertices = vertices_xyz[:, [2, 1, 0]]

# vtkCellArray format:
# [3, i0, i1, i2,
#  3, i0, i1, i2,
#  ...]
polys = vtk_to_numpy(mesh.GetPolys().GetData())
faces = polys.reshape(-1, 4)[:, 1:]


# ------------------------------------------------------------
# Napari
# ------------------------------------------------------------

viewer = napari.Viewer()

viewer.add_labels(
    full_segmentation.astype(np.uint8),
    scale=(10, 2, 2),
    name="segmentation",
)

viewer.add_surface(
    (vertices, faces),
    name="smoothed surface",
    shading="smooth",
)

C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_1528\57995078.py:67: DeprecationWarning: Call to deprecated method GetData. (Use ExportLegacyFormat, or GetOffsetsArray/GetConnectivityArray instead.) -- Deprecated since version 9.6.0.
  polys = vtk_to_numpy(mesh.GetPolys().GetData())


<Surface layer 'smoothed surface' at 0x1c4f9e70dc0>

In [ ]:
faces


array([[  217924,        1,   217925],
       [       1,   217924,        0],
       [  217925,        2,   217926],
       ...,
       [12699796, 12715537, 12699797],
       [12715537, 12699744, 12699797],
       [12699744, 12715537, 12715534]], shape=(25197430, 3))

In [ ]:
import napari
viewer = napari.Viewer()
viewer.add_labels(full_segmentation.astype(np.uint8), scale =(10,2,2))

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


<Labels layer 'Labels' at 0x1c1a32546d0>